# [Open in Colab](https://colab.research.google.com/github/dmochow/BME-i9400-2026/blob/main/notebooks/meeting11_regularization_cross_validation.ipynb)

# BME i9400 — Meeting 11
## Regularisation and Cross-Validation

**Fall 2026 · Wednesday, October 21**

Today's lecture is about controlling and measuring how well a model generalises. It decomposes the
expected error into bias and variance, introduces the L1 penalty alongside the L2 penalty of
Meeting 7, replaces the single validation set with cross-validation, and shows how leakage inside
cross-validation produces results that are wrong.

**Agenda**

1. Bias and variance
2. Ridge regression and the trade-off
3. The lasso
4. Regularised logistic regression
5. Cross-validation
6. Choosing $\lambda$ by cross-validation
7. Leakage inside cross-validation
8. Connection to the course

---
## 1 — Bias and variance

A model is fitted to one training set, but the training set is a random sample: a different cohort
would give different weights and different predictions. The expected error of a model can be
divided according to where it comes from.

The setting:

- the target is $y = f(x) + \varepsilon$
    - $f(x)$ is the true relationship between the features and the target
    - $\varepsilon$ is noise with mean $0$ and variance $\sigma^2$, independent of everything else
- $\hat f(x)$ is the prediction of a model fitted to a random training set $D$
- $\bar f(x) = \mathbb{E}_D\big[\hat f(x)\big]$ is the average prediction over all possible training sets

> $\mathbb{E}_D[\cdot]$ is an **expectation over training sets**: the average of a quantity over every
> training set that could have been drawn, weighted by how likely each is. It is the average that
> the simulation in section 2 approximates by drawing many training sets.

### The decomposition

At a fixed $x$, for a new sample $y$ that was not in the training set, expand the expected squared
error by adding and subtracting $f$:

$$\mathbb{E}\big[(y - \hat f)^2\big] = \mathbb{E}\big[(f + \varepsilon - \hat f)^2\big] = \mathbb{E}\big[\varepsilon^2\big] + 2\,\mathbb{E}\big[\varepsilon\,(f - \hat f)\big] + \mathbb{E}\big[(f - \hat f)^2\big]$$

The new sample's noise is independent of the training set, so the middle term is
$2\,\mathbb{E}[\varepsilon]\,\mathbb{E}[f - \hat f] = 0$, and $\mathbb{E}[\varepsilon^2] = \sigma^2$:

$$\mathbb{E}\big[(y - \hat f)^2\big] = \sigma^2 + \mathbb{E}\big[(f - \hat f)^2\big]$$

Now add and subtract $\bar f$ inside the last term:

$$\mathbb{E}\big[(f - \hat f)^2\big] = \mathbb{E}\big[(f - \bar f + \bar f - \hat f)^2\big] = (f - \bar f)^2 + 2\,(f - \bar f)\,\mathbb{E}\big[\bar f - \hat f\big] + \mathbb{E}\big[(\hat f - \bar f)^2\big]$$

$\mathbb{E}[\hat f] = \bar f$ by definition, so the middle term is again $0$.

### The three terms

$$\boxed{\;\mathbb{E}\big[(y - \hat f)^2\big] \;=\; \underbrace{\sigma^2}_{\text{noise}} \;+\; \underbrace{\big(f - \bar f\big)^2}_{\text{bias}^2} \;+\; \underbrace{\mathbb{E}\big[(\hat f - \bar f)^2\big]}_{\text{variance}}\;}$$

- **Noise**, $\sigma^2$: the part of $y$ that the features cannot predict. No model removes it.
- **Bias**: how far the average prediction is from the truth. A model that is too constrained to
  represent $f$ — too few features, too strong a penalty — is wrong in the same direction on every
  training set.
- **Variance**: how much the prediction changes from one training set to the next. A model with many
  weights fitted to few samples follows the noise in its particular training set.

> This is the decomposition behind Meeting 6, section 5, where the scatter of the least squares
> weights grew as the number of patients fell, and behind Meeting 7, section 6, where ridge helped
> most with 30 patients. A penalty accepts some bias in exchange for a larger reduction in variance.

---
## 2 — Ridge regression and the trade-off

The three terms can be measured directly when $f$ is known. The simulation below uses the
diabetes-progression data of Meetings 6 and 7:

- the "true" $f$ is the least squares fit to all 442 patients, on standardised features
- $\sigma = 54$, the standard deviation of that fit's residuals
- 300 training sets of 30 patients are drawn; each patient's target is $f(x) + \varepsilon$ with
  fresh noise
- ridge regression (Meeting 7) is fitted to every training set at each of 19 values of $\lambda$,
  and the predictions at 100 fixed patients are recorded

> $\text{bias}^2$ and variance are computed at each of the 100 patients from the 300 predictions and
> then averaged over the patients. scikit-learn's `Ridge(alpha=λ)` minimises
> $\|y - Xw\|^2 + \lambda\|w\|^2$, the loss of Meeting 7, with the intercept unpenalised.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import load_diabetes
from sklearn.linear_model import LinearRegression, Ridge, Lasso
plt.rcParams.update({"font.size": 12, "figure.dpi": 110})

data     = load_diabetes(scaled=False, as_frame=True)
features = ["age", "sex", "bmi", "bp", "s1", "s2", "s3", "s4", "s5", "s6"]
X, y     = data.frame[features].values, data.frame["target"].values
Z        = (X - X.mean(axis=0)) / X.std(axis=0, ddof=1)

f_true = LinearRegression().fit(Z, y).predict(Z)               # the "true" f at every patient
sigma  = np.std(y - f_true, ddof=11)
print(f"noise standard deviation sigma = {sigma:.1f},  sigma^2 = {sigma**2:.0f}")

In [ ]:
rng      = np.random.default_rng(0)
at       = rng.choice(len(Z), 100, replace=False)                # the 100 patients where error is measured
lambdas  = np.logspace(-1, 3.5, 19)
n, R     = 30, 300
preds    = np.zeros((len(lambdas), R, len(at)))
for r in range(R):
    rows = rng.choice(len(Z), n, replace=False)
    y_r  = f_true[rows] + rng.normal(0, sigma, n)                # a new training set: true f plus fresh noise
    for i, lam in enumerate(lambdas):
        preds[i, r] = Ridge(alpha=lam).fit(Z[rows], y_r).predict(Z[at])

bias2    = ((preds.mean(axis=1) - f_true[at])**2).mean(axis=1)   # (f - f_bar)^2, averaged over patients
variance = preds.var(axis=1).mean(axis=1)                         # E[(f_hat - f_bar)^2], averaged over patients
total    = sigma**2 + bias2 + variance
best     = np.argmin(total)
for i in [0, 6, best, 13, 18]:
    print(f"lambda {lambdas[i]:8.1f}:  bias^2 {bias2[i]:6.0f}   variance {variance[i]:6.0f}   expected error {total[i]:6.0f}")

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4.8))
ax.semilogx(lambdas, bias2,    lw=2.5, color="#2b6cb0", label="bias$^2$")
ax.semilogx(lambdas, variance, lw=2.5, color="#c1272d", label="variance")
ax.semilogx(lambdas, total,    lw=2.5, color="#5b3a8e", label="expected error = $\\sigma^2$ + bias$^2$ + variance")
ax.axhline(sigma**2, ls=":", color="#6e6878", label="noise $\\sigma^2$")
ax.axvline(lambdas[best], ls="--", color="#5b3a8e", lw=1)
ax.set_xlabel("$\\lambda$"); ax.set_ylabel("squared error (progression units$^2$)")
ax.set_title("Ridge regression, 30 training patients: the bias–variance trade-off")
ax.legend(fontsize=10, loc="upper left"); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

With almost no penalty, the variance term is about 1,600 and the bias term about 20: with 11 weights
and 30 patients, least squares is close to unbiased but its predictions change substantially from one
training set to the next. As $\lambda$ grows the variance falls steadily and the bias rises, slowly
at first and then quickly once the penalty begins to shrink the weights that the data determine well.
The expected error is smallest near $\lambda \approx 18$, where the bias has risen by less than 200 and
the variance has fallen by more than 1,100.

> The minimum lies at an intermediate $\lambda$, and it moves with the size of the training set: with
> more patients the variance at small $\lambda$ is smaller, and less penalty is needed. In practice
> $f$ is unknown, these curves cannot be drawn, and $\lambda$ is chosen by estimating the expected
> error on held-out data — which is the purpose of sections 5 and 6.

---
## 3 — The lasso

The **lasso** replaces the squared norm of the ridge penalty with the sum of absolute values:

$$L_{\text{ridge}}(w) = \|y - Xw\|^2 + \lambda \sum_{j=1}^{d} w_j^2 \qquad\qquad L_{\text{lasso}}(w) = \|y - Xw\|^2 + \lambda \sum_{j=1}^{d} |w_j|$$

> $\sum_j w_j^2 = \|w\|_2^2$ is the squared **L2 norm** of the weights, so ridge is also called L2
> regularisation. $\sum_j |w_j| = \|w\|_1$ is the **L1 norm**, so the lasso is L1 regularisation.
> "Lasso" stands for *least absolute shrinkage and selection operator* (Tibshirani, *Journal of the
> Royal Statistical Society B*, 1996).
>
> As with ridge, the intercept is not penalised and the features are standardised first, so that
> the penalty treats every feature alike.

The change in the penalty changes the solution qualitatively: the lasso sets some weights **exactly to
zero**. It fits the model and selects the features in one step.

### Why the L1 penalty produces zeros

Take a single standardised feature $z$, so that $\frac{1}{n}\sum_i z_i^2 = 1$, and write the loss per
sample with a factor $\tfrac12$. Let $\hat w = \frac{1}{n}\sum_i z_i y_i$, which is the least squares
weight. Expanding the squared error,

$$\frac{1}{2n}\sum_i (y_i - w z_i)^2 = \frac{1}{2n}\sum_i y_i^2 - w\,\frac{1}{n}\sum_i z_i y_i + \frac{w^2}{2}\,\frac{1}{n}\sum_i z_i^2 = \text{const} - w\hat w + \frac{w^2}{2} = \tfrac12 (w - \hat w)^2 + \text{const}$$

So each loss is a parabola centred on $\hat w$, plus a penalty:

| | loss | minimiser |
|---|---|---|
| ridge | $\tfrac12 (w - \hat w)^2 + \tfrac{\lambda}{2} w^2$ | $w = \dfrac{\hat w}{1 + \lambda}$ |
| lasso | $\tfrac12 (w - \hat w)^2 + \lambda\,\lvert w \rvert$ | $w = \operatorname{sign}(\hat w)\,\max\big(\lvert\hat w\rvert - \lambda,\; 0\big)$ |

> Ridge: setting the derivative $w - \hat w + \lambda w$ to zero gives $w(1 + \lambda) = \hat w$.
> The weight is divided by $1 + \lambda$ and is never exactly zero.
>
> Lasso: for $w \neq 0$ the derivative is $w - \hat w + \lambda\,\operatorname{sign}(w) = 0$, so
> $w = \hat w - \lambda\,\operatorname{sign}(w)$. This is consistent only when $|\hat w| > \lambda$.
> When $|\hat w| \le \lambda$, the minimum is at the corner $w = 0$, where $|w|$ has no derivative.
> This rule is called **soft-thresholding**.

In [ ]:
w_hat = np.linspace(-3, 3, 400); lam = 1.0
fig, ax = plt.subplots(figsize=(7.5, 4.6))
ax.plot(w_hat, w_hat, ls=":", color="#6e6878", label="least squares: $w = \\hat w$")
ax.plot(w_hat, w_hat / (1 + lam), lw=2.5, color="#2b6cb0", label="ridge: $\\hat w / (1 + \\lambda)$")
ax.plot(w_hat, np.sign(w_hat) * np.maximum(np.abs(w_hat) - lam, 0), lw=2.5, color="#c1272d",
        label="lasso: soft-threshold at $\\lambda$")
ax.axvspan(-lam, lam, color="#c1272d", alpha=0.08)
ax.set_xlabel("least squares weight $\\hat w$"); ax.set_ylabel("penalised weight $w$")
ax.set_title(f"One standardised feature, $\\lambda$ = {lam:g}"); ax.legend(fontsize=10); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

Ridge shrinks every weight by the same factor. The lasso subtracts a fixed amount from every weight
and sets to zero any weight whose least squares value is smaller than $\lambda$ (the shaded band).
With several correlated features the solution no longer has this closed form, but the behaviour is
the same: as $\lambda$ grows, ridge weights shrink smoothly toward zero together, while lasso weights
reach zero one at a time.

The cell below draws both sets of **coefficient paths** — every weight as a function of the penalty
— for the ten diabetes features of Meeting 7.

> scikit-learn's `Lasso(alpha=α)` minimises $\frac{1}{2n}\|y - Xw\|^2 + \alpha\|w\|_1$, the per-sample
> form used above. Its $\alpha$ is therefore on a different scale from the $\lambda$ of `Ridge`, and
> the two horizontal axes cannot be compared number for number.

In [ ]:
ridge_lams = np.logspace(-2, 4.5, 60)
lasso_alps = np.logspace(-2, 1.8, 60)
W_ridge = np.array([Ridge(alpha=a).fit(Z, y).coef_ for a in ridge_lams])
W_lasso = np.array([Lasso(alpha=a, max_iter=50_000).fit(Z, y).coef_ for a in lasso_alps])

for a in [1, 5, 10, 20]:
    w = Lasso(alpha=a, max_iter=50_000).fit(Z, y).coef_
    kept = [f for f, v in zip(features, w) if v != 0]
    print(f"lasso, alpha = {a:2d}:  {len(kept):2d} non-zero weights  {kept}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4.8), sharey=True)
for j, name in enumerate(features):
    lw = 2.6 if name in ("s1", "s2") else 1.4
    axes[0].semilogx(ridge_lams, W_ridge[:, j], lw=lw, label=name)
    axes[1].semilogx(lasso_alps, W_lasso[:, j], lw=lw, label=name)
axes[0].set_title("Ridge (L2): weights shrink together"); axes[0].set_xlabel("$\\lambda$  (Ridge)")
axes[1].set_title("Lasso (L1): weights reach zero one at a time"); axes[1].set_xlabel("$\\alpha$  (Lasso)")
axes[0].set_ylabel("weight (standardised features)"); axes[1].legend(ncol=2, fontsize=9)
for ax in axes:
    ax.axhline(0, color="k", lw=0.8); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

The lasso paths show the selection directly. The collinear pair of Meeting 7 is resolved by
discarding one member first: `s2` is set to zero near $\alpha = 0.3$, while `s1` survives until
about $\alpha = 3$. At $\alpha = 10$ four
features remain — BMI, blood pressure, `s3`, and `s5` — and at $\alpha = 20$ three. BMI and `s5`,
the features that carried the largest weights in Meeting 7, are the last to leave.

Two properties to keep in mind:

- **Selection among correlated features is unstable.** When two features carry the same information,
  the lasso tends to keep one and discard the other, and which one it keeps can change with a small
  change in the data. A zero weight means the feature was not needed *given the others*, not that it
  is unrelated to the outcome.
- **Elastic net** combines the two penalties, $\lambda_1\|w\|_1 + \lambda_2\|w\|_2^2$. It keeps the
  lasso's zeros while sharing weight across a group of correlated features, as ridge does.

---
## 4 — Regularised logistic regression

The penalties apply unchanged to classification: the cross-entropy loss of Meeting 8 replaces the
squared error.

$$L(w) = -\sum_{i=1}^{n} \Big[\, y_i \log p_i + (1 - y_i) \log (1 - p_i) \,\Big] + \lambda\, \text{penalty}(w), \qquad p_i = \sigma\big(x_i^{\top} w\big)$$

> scikit-learn's `LogisticRegression` writes the same trade-off with $C = 1/\lambda$ multiplying the
> data term rather than the penalty:
>
> $$C \sum_i \text{cross-entropy}_i + \text{penalty}(w)$$
>
> **A small $C$ means a strong penalty.** `penalty="l2"` (the default, with $C = 1$) gives ridge
> logistic regression, and `penalty="l1"` with `solver="liblinear"` or `solver="saga"` gives the lasso.
> Meetings 8 to 10 used `penalty=None`, the unpenalised maximum likelihood fit.

The penalty also solves a problem specific to classification. When some weight vector separates the
two classes perfectly — common with many features and few patients — the unpenalised cross-entropy
can be made smaller indefinitely by scaling $w$ up, and the weights diverge. Any $\lambda > 0$ keeps
them finite.

The model for the rest of the lecture is the Meeting 10 pipeline on all 768 women and all eight
measurements — median imputation, standardisation, logistic regression — now with a penalty.

The question for sections 5 and 6: **how strong should the penalty be?** The training loss cannot
answer it, since any penalty raises the training loss (Meeting 7). The answer has to come from data
that the weights were not fitted to.

In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

URL = "https://raw.githubusercontent.com/dmochow/BME-i9400-2026/main/data/diabetes.csv"
pima = pd.read_csv(URL)
measured = ["Glucose", "BloodPressure", "SkinThickness", "Insulin", "BMI"]
pima[measured] = pima[measured].replace(0, np.nan)               # Meeting 10, section 1
pima_features  = list(pima.columns[:8])
Xp, yp = pima[pima_features], pima["Outcome"].values

def clinical_model(C=None, penalty="l1"):
    # The Meeting 10 pipeline. C=None gives the unpenalised fit; otherwise an L1 penalty of strength 1/C.
    clf = (LogisticRegression(penalty=None, max_iter=1000) if C is None else
           LogisticRegression(penalty=penalty, C=C, solver="liblinear", max_iter=1000))
    return make_pipeline(SimpleImputer(strategy="median"), StandardScaler(), clf)

print(f"{len(yp)} women, {yp.sum()} with diabetes, {len(pima_features)} measurements")

---
## 5 — Cross-validation

Meeting 10 chose a model with one validation set of 154 women. The cell below repeats Meeting 10's
evaluation of the unpenalised eight-feature model on 100 different random splits of the same 768
women. Nothing changes from one split to the next except which women land in the held-out set.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

single_split = []
for seed in range(100):
    X_tr, X_ho, y_tr, y_ho = train_test_split(Xp, yp, test_size=0.2, random_state=seed, stratify=yp)
    p_ho = clinical_model().fit(X_tr, y_tr).predict_proba(X_ho)[:, 1]
    single_split.append(roc_auc_score(y_ho, p_ho))
single_split = np.array(single_split)

print(f"held-out AUC over 100 random 80/20 splits:")
print(f"  mean {single_split.mean():.3f}   standard deviation {single_split.std():.3f}   "
      f"range {single_split.min():.3f} to {single_split.max():.3f}")

The same model on the same data receives a held-out AUC anywhere from about $0.76$ to $0.90$,
depending only on the split. Two models whose true AUCs differ by $0.01$, as in Meeting 10, cannot be
reliably told apart this way. The single held-out set is also wasteful: 20% of the women never
contribute to the fit, and 80% never contribute to the evaluation.

### $K$-fold cross-validation

1. Divide the $n$ samples at random into $K$ groups of nearly equal size, called **folds**
   $F_1, \dots, F_K$.
2. For each $k = 1, \dots, K$:
    - fit the model to every sample **except** those in $F_k$
    - evaluate it on $F_k$, giving a score $s_k$
3. Report the average:
$$\text{CV} = \frac{1}{K}\sum_{k=1}^{K} s_k$$

Every sample is used for evaluation exactly once and for fitting $K - 1$ times. $K = 5$ and $K = 10$
are the usual choices.

> **Stratified** $K$-fold keeps the prevalence the same in every fold, as `stratify=y` did for a single
> split. `StratifiedKFold(5, shuffle=True, random_state=...)` defines the folds; `cross_val_score`
> runs steps 2 and 3 and returns the $K$ scores.
>
> Cross-validation evaluates a **procedure** — "impute, standardise, fit a logistic regression with
> this penalty" — not one fitted model. The $K$ models it fits are discarded. The model that is used
> afterwards is fitted once more, to all of the data.

In [ ]:
from sklearn.model_selection import StratifiedKFold, cross_val_score

folds  = StratifiedKFold(n_splits=5, shuffle=True, random_state=9400)
scores = cross_val_score(clinical_model(), Xp, yp, cv=folds, scoring="roc_auc")
print("AUC on each of the five folds:", np.round(scores, 3))
print(f"5-fold cross-validated AUC:   {scores.mean():.3f}")

cv_estimates = np.array([
    cross_val_score(clinical_model(), Xp, yp, scoring="roc_auc",
                    cv=StratifiedKFold(5, shuffle=True, random_state=seed)).mean()
    for seed in range(100)])
print(f"\nover 100 random partitions into five folds:  standard deviation {cv_estimates.std():.3f}   "
      f"range {cv_estimates.min():.3f} to {cv_estimates.max():.3f}")

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4.4))
bins = np.linspace(0.74, 0.92, 37)
ax.hist(single_split, bins=bins, color="#6e6878", alpha=0.6, label="one 80/20 split (100 random splits)")
ax.hist(cv_estimates, bins=bins, color="#5b3a8e", alpha=0.85, label="5-fold cross-validation (100 random partitions)")
ax.set_xlabel("estimated AUC of the same procedure"); ax.set_ylabel("count")
ax.set_title("How much the estimate depends on which women are held out"); ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

The five fold AUCs themselves differ by almost $0.05$, as single splits do. Their average depends far
less on the partition: across 100 partitions the five-fold estimate stays within about $\pm 0.01$,
against $\pm 0.07$ for a single split.

> The narrow purple histogram shows how much the estimate depends on **the partition**. It is not
> the uncertainty of the estimate. Every partition uses the same 768 women, and a different cohort of
> 768 would give a different answer; that uncertainty is the subject of Meeting 20.

---
## 6 — Choosing $\lambda$ by cross-validation

Cross-validation replaces the validation set of Meeting 7 in choosing a hyperparameter: run it once
for every candidate value and keep the value with the best cross-validated score. The cell below
does this for the L1-penalised clinical model over 14 values of $C$, using the same five folds for
every value so that the comparison is fair.

> The **standard error** of the cross-validated score is estimated from the spread of the $K$ fold
> scores, $\text{SE} = \text{SD}(s_1, \dots, s_K)/\sqrt{K}$. It is a rough guide: the folds share
> training data, so the scores are not independent.

In [ ]:
Cs = np.logspace(-2, 1, 14)
cv_mean, cv_se, n_nonzero = [], [], []
for C in Cs:
    s = cross_val_score(clinical_model(C), Xp, yp, cv=folds, scoring="roc_auc")
    cv_mean.append(s.mean()); cv_se.append(s.std(ddof=1) / np.sqrt(len(s)))
    n_nonzero.append(np.sum(clinical_model(C).fit(Xp, yp)[-1].coef_ != 0))   # weights kept, fitted to all 768
cv_mean, cv_se, n_nonzero = map(np.array, (cv_mean, cv_se, n_nonzero))

i_best = np.argmax(cv_mean)
i_1se  = np.where(cv_mean >= cv_mean[i_best] - cv_se[i_best])[0][0]   # smallest C within one SE of the best
print(f"best C          = {Cs[i_best]:.3f}:  CV AUC {cv_mean[i_best]:.3f} (SE {cv_se[i_best]:.3f}),  {n_nonzero[i_best]} measurements kept")
print(f"one-SE rule     = {Cs[i_1se]:.3f}:  CV AUC {cv_mean[i_1se]:.3f},  {n_nonzero[i_1se]} measurements kept")

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4.6))
ax.errorbar(Cs, cv_mean, yerr=cv_se, fmt="o-", color="#5b3a8e", lw=2, capsize=3, label="5-fold CV AUC $\\pm$ 1 SE")
ax.axvline(Cs[i_best], ls="--", color="#5b3a8e", lw=1, label="best C")
ax.axvline(Cs[i_1se],  ls="--", color="#c1272d", lw=1.5, label="one-SE rule")
ax.set_xscale("log"); ax.set_xlabel("C   (stronger penalty to the left)"); ax.set_ylabel("cross-validated AUC")
ax2 = ax.twiny(); ax2.set_xscale("log"); ax2.set_xlim(ax.get_xlim())
ax2.set_xticks(Cs[::2]); ax2.set_xticklabels(n_nonzero[::2]); ax2.set_xlabel("measurements kept")
ax.set_title("L1-penalised logistic regression, all 768 women", pad=34); ax.legend(loc="lower right"); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

In [ ]:
final = clinical_model(Cs[i_1se]).fit(Xp, yp)            # the chosen procedure, refitted to all of the data
print(f"final model: L1 penalty, C = {Cs[i_1se]:.3f}, fitted to all {len(yp)} women\n")
for name, w in zip(pima_features, final[-1].coef_[0]):
    print(f"  {name:<26} {w:+.2f}" + ("" if w != 0 else "   (removed)"))

The cross-validated AUC is flat at about $0.84$ for every $C$ above about $0.05$ and falls for
stronger penalties, to $0.79$ at $C = 0.01$, where only glucose remains. The best value, $C \approx 0.14$,
already removes three measurements. The **one-standard-error rule** chooses instead the strongest
penalty whose score is within one standard error of the best: the data cannot distinguish it from the
best, and its weights are smaller. Here it gives $C \approx 0.05$, with the same five measurements.
Both remove blood pressure, skin-fold thickness, and insulin — the lasso reaches the conclusion that
Meeting 10 reached by comparing three hand-built models.

One problem remains. The score at the chosen $C$ is the **maximum** of 14 cross-validated scores,
each computed on the same folds, and a maximum of noisy estimates is optimistic — the same optimism as
the threshold of Meeting 10, now in the hyperparameter. An unbiased estimate needs a second, outer
layer of held-out data around the whole selection procedure: **nested cross-validation**, the
subject of Meeting 12.

---
## 7 — Leakage inside cross-validation

Cross-validation is unbiased only if each fold's held-out samples play no part in fitting the model
that is evaluated on them. **Every step that learns from data is part of the model** — imputation
medians, standardisation statistics, feature selection, hyperparameter choice — and must be refitted
inside each fold, on that fold's training samples only. When a step is fitted once to all of the
data and cross-validation is run afterwards, information about the held-out samples has leaked into
the model.

The cell below constructs a dataset with no signal at all:

- 60 patients and 5,000 features, every feature drawn from a standard normal distribution
- labels assigned by a coin flip: 30 patients in each class, independent of every feature

The correct cross-validated AUC is $0.5$.

Two analyses of the same data, each selecting the 20 features most associated with the label and
fitting a logistic regression to them:

- **the wrong way:** select the 20 features using all 60 patients, then cross-validate the logistic
  regression on those 20 features
- **the right way:** put the selection inside a pipeline, so that within each fold the 20 features
  are chosen from that fold's 48 training patients only

> `SelectKBest(f_classif, k=20)` scores every feature by an F-statistic comparing its mean in the two
> classes — for two classes, the square of the two-sample $t$-statistic — and keeps the 20 highest.
> This is the "top differentially expressed genes" step of many published biomarker studies.

In [ ]:
from sklearn.feature_selection import SelectKBest, f_classif

rng = np.random.default_rng(9400)
X_noise = rng.normal(size=(60, 5000))
y_noise = rng.permutation(np.repeat([0, 1], 30))                # coin-flip labels
cv5 = StratifiedKFold(5, shuffle=True, random_state=9400)

# wrong: selection sees every patient, including those that will be held out
X_sel = SelectKBest(f_classif, k=20).fit_transform(X_noise, y_noise)
wrong = cross_val_score(LogisticRegression(max_iter=1000), X_sel, y_noise, cv=cv5, scoring="roc_auc")

# right: selection is refitted inside each fold
pipe  = make_pipeline(SelectKBest(f_classif, k=20), LogisticRegression(max_iter=1000))
right = cross_val_score(pipe, X_noise, y_noise, cv=cv5, scoring="roc_auc")

print(f"selection outside cross-validation:  CV AUC {wrong.mean():.3f}")
print(f"selection inside cross-validation:   CV AUC {right.mean():.3f}")

In [ ]:
# The same comparison on 50 independent noise datasets.
wrong_all, right_all = [], []
for seed in range(50):
    r  = np.random.default_rng(seed)
    Xn = r.normal(size=(60, 5000)); yn = r.permutation(np.repeat([0, 1], 30))
    wrong_all.append(cross_val_score(LogisticRegression(max_iter=1000),
                                     SelectKBest(f_classif, k=20).fit_transform(Xn, yn), yn, cv=cv5, scoring="roc_auc").mean())
    right_all.append(cross_val_score(pipe, Xn, yn, cv=cv5, scoring="roc_auc").mean())
print(f"selection outside: mean CV AUC {np.mean(wrong_all):.3f}  (lowest of 50: {np.min(wrong_all):.3f})")
print(f"selection inside:  mean CV AUC {np.mean(right_all):.3f}")

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4.2))
bins = np.linspace(0.2, 1.0, 33)
ax.hist(right_all, bins=bins, color="#2b6cb0", alpha=0.8, label="selection inside each fold")
ax.hist(wrong_all, bins=bins, color="#c1272d", alpha=0.8, label="selection before cross-validation")
ax.axvline(0.5, ls="--", color="k", lw=1.2, label="the true AUC: labels are coin flips")
ax.set_xlabel("cross-validated AUC"); ax.set_ylabel("datasets (of 50)")
ax.set_title("60 patients, 5,000 noise features, random labels"); ax.legend(loc="upper left"); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

With the selection outside cross-validation, a classifier of pure noise receives a cross-validated
AUC near $0.99$ on every one of the 50 datasets. Among 5,000 random features, some are associated with
the labels of these 60 patients by chance; selecting them with all 60 patients' labels guarantees that
the held-out patients' labels were used to build the model. With the selection inside the folds, the
estimates scatter around $0.5$, as they must.

> This error has appeared in the biomedical literature. Ambroise and McLachlan (*PNAS*, 2002)
> re-analysed published microarray classifiers and showed that error rates near zero, reported after
> selecting genes on the full dataset, rose to chance levels when the selection was done inside the
> cross-validation.

### Where leakage enters

| Source | Example | Remedy |
|---|---|---|
| preprocessing fitted to all data | imputation medians, standardisation (Meeting 10) — usually small | put the step in the `Pipeline` |
| feature selection on all data | the example above — can be very large | put the step in the `Pipeline` |
| tuning on the evaluation folds | reporting the best of 14 CV scores (section 6) | nested cross-validation (Meeting 12) |
| several samples per patient | repeated visits, several images or ECG beats per patient, split across folds | `GroupKFold`: all of a patient's samples in one fold |
| time order | training on later patients, testing on earlier ones | train on the past, test on the future |

> A **group** is a set of samples that are not independent, most often the samples from one patient.
> If a patient's samples are divided between training and held-out folds, the model can recognise
> the patient rather than the disease. `GroupKFold` and `StratifiedGroupKFold` take a `groups` array
> of patient identifiers and keep each group whole.

The rule covering every row: **pass the whole procedure, as a `Pipeline`, to the cross-validation**,
and split by patient whenever a patient contributes more than one sample.

---
## 8 — Connection to the course

**Meetings 6 and 7, explained.** The growing scatter of the least squares weights with few patients
is the variance term; ridge's gain with 30 patients and its absence with 353 is the trade-off of
section 2. The lasso adds a second effect, selection, which Meeting 10 carried out by hand.

**Meeting 10, made more reliable.** One validation set of 154 women has been replaced by five folds
that use all 768. The test-set rule of Meeting 10 still holds: the cross-validated score of the
chosen $\lambda$ is optimistic, and Meeting 12 adds the outer loop that removes the optimism.

**Later in the course.** Every model from here on — the neural networks of Meeting 17, the ensembles
of Meeting 19, the image encoders of Meeting 22 — has hyperparameters that are chosen in the same way
and a preprocessing procedure that is subject to the same leakage.

## What to take away

- Expected error = noise + bias$^2$ + variance; a penalty adds bias and removes variance, and helps
  most when samples are few
- Ridge (L2) shrinks all weights together; the lasso (L1) sets weights exactly to zero and so
  selects features; standardise first and leave the intercept unpenalised
- In scikit-learn's `LogisticRegression`, $C = 1/\lambda$: a small $C$ is a strong penalty
- $K$-fold cross-validation evaluates every sample once and depends far less on the partition than a
  single split; it evaluates a procedure, which is then refitted to all of the data
- Choose $\lambda$ by cross-validation; the one-standard-error rule prefers the simpler model; the
  best CV score is optimistic
- Every step that learns from data goes inside the folds, as a `Pipeline`; split by patient

### Monday, October 26 — Studio

Nested cross-validation and calibration: an outer loop that gives an unbiased estimate of the whole
selection procedure of section 6, and a check of whether the model's probabilities can be read as
probabilities.

## Recommended resources

Everything here is optional and freely available unless marked **(CCNY)**. Each item covers material
from this lecture in a different style or voice; none of it is assigned.

### Start here

| Resource | Why this one | Time |
|---|---|---|
| **[Machine Learning Fundamentals: Bias and Variance](https://www.youtube.com/watch?v=EuBBz3bI-aA)** — StatQuest | Section 1 without equations. | 6 min |
| **[Regularization Part 1: Ridge (L2) Regression](https://www.youtube.com/watch?v=Q81RR3yKn30)** — StatQuest | Ridge as a trade of bias for variance, with small examples. | 20 min |
| **[Regularization Part 2: Lasso (L1) Regression](https://www.youtube.com/watch?v=NGf0voTMlcs)** — StatQuest | Why the lasso removes features and ridge does not. | 8 min |
| **[Machine Learning Fundamentals: Cross Validation](https://www.youtube.com/watch?v=fSytzGwwBVw)** — StatQuest | The $K$-fold procedure of section 5. | 6 min |

### Going further

| Resource | Why this one | Time |
|---|---|---|
| **[An Introduction to Statistical Learning](https://www.statlearning.com/)** — James, Witten, Hastie, Tibshirani, chapters 5 and 6 | Cross-validation, ridge, and the lasso at the level of this course, with Python labs. Free PDF. | 3 h |
| **[The Elements of Statistical Learning](https://hastie.su.domains/ElemStatLearn/)** — Hastie, Tibshirani, Friedman, section 7.10.2 | "The wrong and right way to do cross-validation": the example of section 7. Free PDF. | 20 min |
| **[Common pitfalls and recommended practices](https://scikit-learn.org/stable/common_pitfalls.html)** — scikit-learn User Guide | Data leakage and the `Pipeline` remedy, with code. | 20 min |
| **[Selection bias in gene extraction on the basis of microarray gene-expression data](https://doi.org/10.1073/pnas.102102699)** — Ambroise and McLachlan, *PNAS*, 2002 | The published-literature case of section 7. Open access. | 45 min |
| **[Leakage and the reproducibility crisis in machine-learning-based science](https://doi.org/10.1016/j.patter.2023.100804)** — Kapoor and Narayanan, *Patterns*, 2023 | A survey of leakage across 17 fields, with a taxonomy. Open access. | 1 h |
| **[Regression Shrinkage and Selection via the Lasso](https://doi.org/10.1111/j.2517-6161.1996.tb02080.x)** — Tibshirani, *JRSS B*, 1996 **(CCNY)** | The original lasso paper. | 1.5 h |